<div style="border-radius: 10px; padding: 32px 0px; border: 1px solid rgba(128,128,128,0.2);">

  <div style="display: flex; justify-content: space-between; align-items: flex-start; flex-wrap: wrap; gap: 16px; padding: 0px 32px;">

  <div>
    <div style="font-size: 0.75rem; letter-spacing: 3px; text-transform: uppercase; font-weight: 600; margin-bottom: 10px; opacity: 0.6;">
      Máster Universitario en Big Data y Computación en la Nube.
    </div>
    <div style="font-size: 1.5rem; font-weight: 700; margin-bottom: 4px;">
      Trabajo de Fin de Máster
    </div>
    <div style="font-size: 1rem; font-weight: 400; opacity: 0.75;">
      Clasificador taxonómico de boletines oficiales españoles
    </div>
  </div>

  <div style="margin-top: 16px; display: flex; align-items: center; gap: 12px;">
    <div style="font-size: 1rem; font-weight: 600;">Hugo de Lamo</div>
  </div>

  </div>
</div>

# 05 · Clasificador taxonómico de boletines oficiales

Este notebook implementa un clasificador multietiqueta de publicaciones de boletines oficiales españoles usando **Pydantic AI**. El problema es una aguja en un pajar: de ~65 000 publicaciones del Q1 2025, solo ~3,7 % son relevantes para el dominio ambiental-energético.

El clasificador responde cuatro preguntas por publicación:
1. **¿Es relevante?** — ¿Pertenece al universo de autorizaciones ambiental-energéticas?
2. **¿Qué procedimientos contiene?** — Lista multilabel: DIA, AAP, AAC, AAU, IIA, AAI, IAE, DUP.
3. **¿Qué tipo de acto es?** — Forma jurídica del documento (N1): resolución, anuncio, decreto…
4. **¿Qué tecnología menciona?** — Lista multilabel: fotovoltaica, eólica, hidrógeno…

---

## Estructura del notebook

| § | Sección | Contenido |
|---|---------|----------|
| **0** | **Setup** | Entorno, dependencias, modelo |
| **1** | **Schema de output** | `ClassifierOutput`, enums y reglas de negocio |
| **2** | **Ground truth** | Construcción del dataset etiquetado manualmente |
| **3** | **Agente clasificador** | System prompt, agent con Pydantic AI, run single |
| **4** | **Baseline** | Experimento 0 — claude-sonnet-4-6 sobre ground truth |
| **5** | **Experimentos de ablación** | Few-shot, `act_type` y campo `technologies` |
| **6** | **Modelos locales** | Experimento 4 — modelos 7B via LM Studio |
| **7** | **Análisis y conclusiones** | Comparativa de métricas, falsos positivos, siguientes pasos |

---

## §0. Setup

Cargamos las variables de entorno (la API key de Gemini vive en `.env`, nunca en el código) e importamos las librerías del proyecto.

In [ ]:
import os
from pathlib import Path

import pandas as pd
from dotenv import find_dotenv, load_dotenv
from pydantic_ai import Agent

load_dotenv(find_dotenv())

In [ ]:
# Único punto de cambio para conectar otro proveedor.
# Para LM Studio (modelos locales): "openai:nombre-del-modelo" con base_url="http://127.0.0.1:1234/v1"
MODEL = "gemini-2.5-pro"

In [ ]:
api_key = os.getenv("GOOGLE_API_KEY")
assert api_key, "GOOGLE_API_KEY no encontrada — revisa el archivo .env"

print("Entorno listo.")
print(f"  Modelo:  {MODEL}")
print(f"  API key: {api_key[:8]}{'*' * (len(api_key) - 8)}")

---

## §1. Schema de output

El schema define el **contrato entre el LLM y el sistema**: qué campos devuelve el modelo, de qué tipo y bajo qué restricciones. Pydantic valida cada respuesta antes de que llegue al resto del código, forzando un retry automático si algo no cumple el schema.

`ClassifierOutput` tiene 6 campos:

| Campo | Tipo | Rol |
|-------|------|-----|
| `is_relevant` | `bool` | ¿Pertenece al dominio ambiental-energético? |
| `act_type` | `ActType` | Forma jurídica del acto (N1) — resolución, anuncio, decreto… |
| `procedures` | `list[ProcedureType]` | Procedimientos identificados (N2) — multilabel |
| `technologies` | `list[TechnologyType]` | Tecnologías mencionadas (N3) — multilabel, puede ser vacía |
| `confidence` | `float` | Confianza global entre 0.0 y 1.0 |
| `reasoning` | `str` | Justificación breve citando el texto que dispara cada etiqueta |

Un `@model_validator` impone los invariantes de negocio: `is_relevant=True` exige `procedures != []`; `is_relevant=False` exige ambas listas vacías.

In [ ]:
from clasificador.schema import ActType, ProcedureType, TechnologyType, ClassifierOutput

In [ ]:
# Ejemplo válido: resolución de DIA con tecnología fotovoltaica
ejemplo_valido = ClassifierOutput(
    is_relevant=True,
    act_type=ActType.RESOLUCION,
    procedures=[ProcedureType.DIA],
    technologies=[TechnologyType.FOTOVOLTAICA],
    confidence=0.98,
    reasoning=(
        "'se formula la declaración de impacto ambiental' → DIA en resolución. "
        "'Planta Solar Fotovoltaica' → fotovoltaica."
    ),
)
print("Ejemplo válido:")
print(ejemplo_valido.model_dump_json(indent=2))

# Ejemplo que viola el invariante: is_relevant=False con procedures no vacío
print("\nViolación de invariante:")
try:
    ClassifierOutput(
        is_relevant=False,
        act_type=ActType.RESOLUCION,
        procedures=[ProcedureType.AAP],
        technologies=[],
        confidence=0.5,
        reasoning="Prueba de invariante.",
    )
except Exception as e:
    print(f"  ValidationError capturado → {e.errors()[0]['msg']}")

---

## §2. Ground truth

El ground truth tiene tres capas que se construyen en orden:

| Capa | Qué etiqueta | Cómo | Estado |
|------|-------------|------|--------|
| **1 — N1 determinista** | `act_type` | Reglas de primer token sobre `description` | Esta sección |
| **2 — N2 por LLM** | `procedures`, `technologies`, `is_relevant` | Agente Pydantic AI sobre muestra estratificada | §3–§4 |
| **3 — Revisión manual** | Correcciones y casos límite | Inspección humana de los desacuerdos | Post-baseline |

Esta sección implementa la **Capa 1**: inferencia determinista de `act_type` mediante reglas de primer token. El resultado se usará como señal de entrada al prompt del LLM en §3 y como columna de estratificación para el muestreo del ground truth en §4.

In [ ]:
import html
import re

PATH_PARQUET = "../data/raw/silver_official_gazettes_2025_Q1.parquet"

df = pd.read_parquet(PATH_PARQUET)
df["description"] = df["description"].apply(html.unescape)

print(f"Corpus: {len(df):,} registros · {df['bulletin'].nunique()} boletines")

In [ ]:
def preprocess_description(desc: str, bulletin: str) -> str:
    desc = desc.strip()
    # BOCM: "Tema\n– Tipo de acto..."
    if "\n–" in desc:
        desc = desc.split("\n–", 1)[1].strip()
    elif "\n-" in desc:
        desc = desc.split("\n-", 1)[1].strip()
    # BOCA: "Organismo.- Tipo de acto..."
    if bulletin == "boca" and ".-" in desc:
        desc = desc.split(".-", 1)[1].strip()
    # BOE topónimos: descripción es solo ciudad en mayúsculas
    if re.match(r"^[A-ZÁÉÍÓÚÜÑ/\s]+$", desc) and len(desc.split()) <= 4:
        return "__TOPONIMO__"
    # BOE subastas AEAT
    if desc.upper().startswith(("U.R.", "E.R.", "SUMA GESTIÓN", "ORGANISMO AUTÓNOMO DE HACIENDA")):
        return "__SUBASTA_AEAT__"
    return desc

In [ ]:
_N1_MAP = [
    (r"corrección de errat",             ActType.CORRECCION_ERRORES),
    (r"corrección de error",             ActType.CORRECCION_ERRORES),
    (r"rectificación",                   ActType.CORRECCION_ERRORES),
    (r"real decreto",                    ActType.REAL_DECRETO),
    (r"orden foral",                     ActType.ORDEN),
    (r"información pública",             ActType.INFORMACION_PUBLICA),
    (r"exposición pública",              ActType.INFORMACION_PUBLICA),
    (r"trámite de información",          ActType.INFORMACION_PUBLICA),
    (r"resolución",                      ActType.RESOLUCION),
    (r"anuncio",                         ActType.ANUNCIO),
    (r"orden",                           ActType.ORDEN),
    (r"decreto foral",                   ActType.DECRETO),
    (r"decreto",                         ActType.DECRETO),
    (r"acuerdo",                         ActType.ACUERDO),
    (r"aprobación",                      ActType.APROBACION),
    (r"extracto",                        ActType.EXTRACTO),
    (r"convenio",                        ActType.CONVENIO),
    (r"adenda",                          ActType.CONVENIO),
    (r"solicitud",                       ActType.SOLICITUD),
    (r"modificación",                    ActType.MODIFICACION),
    (r"edicto",                          ActType.EDICTO),
    (r"notificación",                    ActType.NOTIFICACION),
    (r"notificaciones",                  ActType.NOTIFICACION),
    (r"recaudación ejecutiva",           ActType.NOTIFICACION),
    (r"propuesta de resolución",         ActType.RESOLUCION),
    (r"bases",                           ActType.CONVOCATORIA),
    (r"convocatoria",                    ActType.CONVOCATORIA),
    (r"nombramiento",                    ActType.RESOLUCION),
    (r"delegación",                      ActType.RESOLUCION),
    (r"emplazamiento",                   ActType.NOTIFICACION),
    (r"citación",                        ActType.NOTIFICACION),
    (r"diligencia",                      ActType.NOTIFICACION),
    (r"cédula",                          ActType.NOTIFICACION),
    (r"requerimiento",                   ActType.NOTIFICACION),
    (r"sala primera",                    ActType.OTROS),
    (r"sala segunda",                    ActType.OTROS),
    (r"__toponimo__",                    ActType.OTROS),
    (r"__subasta_aeat__",                ActType.OTROS),
]


def inferir_act_type(description: str, bulletin: str) -> ActType:
    desc_clean = preprocess_description(description, bulletin)
    text = desc_clean.lower().strip()
    for pattern, act_type in _N1_MAP:
        if text.startswith(pattern):
            return act_type
    return ActType.OTROS

In [ ]:
df["act_type_n1"] = df.apply(
    lambda row: inferir_act_type(row["description"], row["bulletin"]), axis=1
)

print("Distribución N1 inferida:\n")
dist = df["act_type_n1"].value_counts()
total = len(df)
for val, count in dist.items():
    print(f"  {val:<25} {count:>6,}  ({count/total*100:.1f}%)")

otros = (df["act_type_n1"] == ActType.OTROS).sum()
cobertura = (1 - otros / total) * 100
print(f"\nCobertura N1: {cobertura:.1f}%  ({otros:,} registros en OTROS)")

### Límites del clasificador N1 y trabajo futuro

El clasificador de primer token cubre ~96% del corpus con reglas deterministas.
El 4% restante cae en `OTROS` por tres motivos distintos con soluciones conocidas:

| Grupo | Volumen | Motivo | Solución futura |
|---|---|---|---|
| Topónimos BOE / subastas AEAT | ~5.600 | Sin contenido textual real — irrecuperable sin PDF | Clasificar como clase propia `NO_INFERIBLE` en v2 |
| BOCM/BOCA residual | ~300 | Variantes de cabecera no contempladas | Ampliar reglas de pre-procesamiento |
| RRHH sin tipo explícito | ~2.000 | `RELACIÓN`, `LISTA`, `OFERTA`, `REGISTRO`, `ASPIRANTES`... — el tipo de acto no aparece en la descripción | LLM zero-shot puede inferirlo desde el contenido; viable en v2 |

El tercer grupo es el más interesante: los registros de RRHH con descripción como
*"Relación definitiva de aspirantes admitidos..."* o *"Lista provisional de admitidos..."*
tienen suficiente contenido semántico para que un LLM infiera `resolución` o `anuncio`.
Esto está documentado aquí como trabajo pendiente para la extensión v2 del clasificador.